# Оценка рейтингов фильмов


**Датасет:** MovieLens 20M — https://www.kaggle.com/datasets/grouplens/movielens-20m-dataset

**Постановка:** гибридное предсказание оценки `user → movie` (регрессия, `rating ∈ [0.5, 5.0]`).

**Метрики качества:** RMSE, MAE, R².

---

## 1. Введение и постановка задачи

### 1.1. Основные понятия

- **Рекомендательная система** — система, предсказывающая предпочтения пользователя
  (в нашем случае — оценку фильма).
- **Коллаборативная фильтрация (collaborative filtering)** — подход, использующий
  историю оценок множества пользователей: похожие пользователи похоже оценивают
  фильмы.
- **Контентная фильтрация (content-based)** — подход, опирающийся на признаки самих
  объектов (жанры, теги, год).
- **Гибридный подход** — объединяет коллаборативную и контентную информацию.
- **Эмбеддинги / латентные факторы (latent factors)** — обучаемые плотные векторы,
  кодирующие пользователей и фильмы в скрытом пространстве.
- **Матричная факторизация (Matrix Factorization)** — приближение матрицы оценок
  произведением матриц латентных факторов пользователей и фильмов.
- **Нейросетевая коллаборативная фильтрация (Neural CF / NeuMF)** — замена скалярного
  произведения эмбеддингов на нелинейную нейросеть (MLP).

### 1.2. Постановка задачи

Дана история оценок $(u, i, r)$, где $u$ — пользователь, $i$ — фильм,
$r \in \{0.5, 1.0, \dots, 5.0\}$ — оценка.

Требуется построить модель $\hat{r} = f(u, i, x_i)$, предсказывающую оценку, где
$x_i$ — контентные признаки фильма (жанры, genome-теги, год выпуска).

Это **задача регрессии**: целевая переменная — вещественная оценка в диапазоне
$[0.5, 5.0]$.

### 1.3. Подходы и методы решения

Рассматриваем последовательность моделей нарастающей сложности:

1. **Baseline** — глобальное среднее и модель смещений
   $\hat{r}_{ui} = \mu + b_u + b_i$ (точка отсчёта).
2. **Matrix Factorization (MF)** — эмбеддинги пользователя и фильма + смещения,
   скалярное произведение.
3. **NeuMF (Neural CF)** — конкатенация эмбеддингов user/movie, поданная в MLP.
4. **Hybrid** — эмбеддинги user/movie + отобранные контентные признаки фильма → MLP.

Сравнение этих методов между собой и с baseline — основная цель эксперимента.

### 1.4. Метрики оценки качества

- **RMSE** (Root Mean Squared Error) — основная метрика, сильнее штрафует крупные
  ошибки:
  $$\mathrm{RMSE} = \sqrt{\frac{1}{N}\sum_{(u,i)} (r_{ui} - \hat{r}_{ui})^2}.$$
- **MAE** (Mean Absolute Error) — средняя абсолютная ошибка, устойчивее к выбросам:
  $$\mathrm{MAE} = \frac{1}{N}\sum_{(u,i)} |r_{ui} - \hat{r}_{ui}|.$$
- **R²** (коэффициент детерминации) — доля объяснённой дисперсии целевой переменной.

Качество измеряем на отложенной (test) выборке, не участвующей в обучении и подборе
гиперпараметров.

## 2. Импорт библиотек и конфигурация

In [ ]:
from pathlib import Path

# Работа с табличными данными
import numpy as np
import pandas as pd

# Визуализация
import matplotlib.pyplot as plt

# PyTorch
import torch
from torch import nn
from torch.utils.data import Dataset, DataLoader

# Метрики качества и разбиение выборки
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.model_selection import train_test_split

In [ ]:
# --- Конфигурация эксперимента ---
# Собираем все настройки в одном месте, чтобы исследование было
# воспроизводимым и его легко было корректировать.

SEED = 42  # фиксированное зерно ГСЧ для воспроизводимости результатов

# Пути к данным
DATA_DIR = Path("data")

# Параметры формирования подвыборки (раздел 3).
# Полный rating.csv содержит ~20 млн оценок, что избыточно для учебного
# стенда. Оставляем только "активных" пользователей и ограничиваем общий
# объём выборки, чтобы ускорить обучение и подбор гиперпараметров.
MIN_USER_RATINGS = 50         # минимальное число оценок у "активного" пользователя
TARGET_RATINGS = 2_000_000    # целевой объём подвыборки (порядка 1–2 млн оценок)

# Параметры предобработки (раздел 5).
MIN_MOVIE_RATINGS = 20        # минимальное число оценок у фильма (отсев "длинного хвоста")
VAL_SIZE = 0.1                # доля валидационной выборки
TEST_SIZE = 0.1               # доля тестовой выборки

# Фиксируем зерно для numpy и torch
np.random.seed(SEED)
torch.manual_seed(SEED)

# Выбираем устройство: GPU при наличии, иначе CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# --- Единый стиль графиков ---
# Все настройки оформления собраны здесь, чтобы все графики работы имели
# строгий, чистый и единообразный академический вид.
plt.rcParams.update({
    "figure.figsize": (9, 5),
    "figure.dpi": 110,
    # Шрифты и заголовки
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.titleweight": "medium",
    "axes.labelsize": 10.5,
    # Рамка осей: убираем верхнюю и правую линии — меньше визуального шума.
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#555555",
    "axes.linewidth": 0.8,
    # Сетка: тонкая, бледная и ВСЕГДА под данными (axisbelow), чтобы она
    # помогала читать значения, но не отвлекала от столбцов и линий.
    "axes.grid": True,
    "axes.axisbelow": True,
    "grid.color": "#b5b5b5",
    "grid.linewidth": 0.4,
    "grid.alpha": 0.35,
    # Столбцы/гистограммы: тонкая светлая обводка для аккуратного разделения.
    "patch.linewidth": 0.7,
    "patch.edgecolor": "white",
    "patch.force_edgecolor": True,
    # Линии и маркеры
    "lines.linewidth": 1.8,
    "lines.markersize": 5,
    # Легенда без рамки
    "legend.frameon": False,
    "xtick.color": "#555555",
    "ytick.color": "#555555",
})

print(f"Версия PyTorch: {torch.__version__}")
print(f"Устройство для обучения: {device}")

## 3. Загрузка данных

Состав датасета MovieLens 20M:

| Файл | Содержание | Используем |
|------|-----------|-----------|
| `rating.csv` | userId, movieId, rating, timestamp (≈20 млн строк) | да — целевые оценки |
| `movie.csv` | movieId, title, genres | да — жанры, год из названия |
| `genome_scores.csv` | movieId, tagId, relevance (≈1128 тегов) | да — контентные признаки |
| `genome_tags.csv` | tagId, tag | да — расшифровка тегов |
| `tag.csv` | пользовательские текстовые теги | нет (опционально) |
| `link.csv` | связи с IMDb/TMDb | нет |

Из-за объёма `rating.csv` (~690 МБ) работаем с **подвыборкой активных
пользователей** (~1–2 млн оценок).

In [ ]:
# --- Загрузка справочников ---

# Описание фильмов: идентификатор, название (с годом), список жанров
movies = pd.read_csv(DATA_DIR / "movie.csv")

# Genome-теги: матрица релевантности "фильм × тег" (~11 млн строк).
# Указываем компактные типы, чтобы снизить расход памяти.
genome_scores = pd.read_csv(
    DATA_DIR / "genome_scores.csv",
    dtype={"movieId": "int32", "tagId": "int16", "relevance": "float32"},
)

# Расшифровка genome-тегов: tagId -> текстовое название тега
genome_tags = pd.read_csv(DATA_DIR / "genome_tags.csv")

print(f"movies:        {movies.shape}")
print(f"genome_scores: {genome_scores.shape}")
print(f"genome_tags:   {genome_tags.shape}")
movies.head()

In [ ]:
# --- Загрузка оценок с подвыборкой активных пользователей ---
# rating.csv — самый большой файл (~20 млн строк, 690 МБ). Чтобы стенд
# оставался лёгким, формируем подвыборку активных пользователей и
# КЭШИРУЕМ её в data/. При повторных запусках ноутбука подвыборка просто
# читается из кэша — тяжёлый разбор всего rating.csv не повторяется.

SAMPLE_PATH = DATA_DIR / "ratings_sample.csv"

if SAMPLE_PATH.exists():
    # Подвыборка уже была сформирована ранее — читаем готовый файл.
    ratings = pd.read_csv(
        SAMPLE_PATH,
        dtype={"userId": "int32", "movieId": "int32", "rating": "float32"},
        parse_dates=["timestamp"],
    )
    print(f"Подвыборка загружена из кэша: {SAMPLE_PATH}")
else:
    # Кэша нет — формируем подвыборку из полного rating.csv (самый долгий шаг).
    ratings_full = pd.read_csv(
        DATA_DIR / "rating.csv",
        dtype={"userId": "int32", "movieId": "int32", "rating": "float32"},
        parse_dates=["timestamp"],
    )
    print(f"Полный размер rating.csv: {len(ratings_full):,} оценок, "
          f"{ratings_full['userId'].nunique():,} пользователей")

    # Шаг 1. Отбираем активных пользователей (>= MIN_USER_RATINGS оценок):
    # редкие пользователи дают мало информации для обучения эмбеддингов.
    user_counts = ratings_full["userId"].value_counts()
    active_users = user_counts.index[user_counts >= MIN_USER_RATINGS].to_numpy()
    print(f"Активных пользователей (>= {MIN_USER_RATINGS} оценок): {len(active_users):,}")

    # Шаг 2. Случайно набираем активных пользователей, пока суммарное число
    # их оценок не достигнет целевого объёма. Это даёт выборку нужного
    # размера без смещения по конкретным пользователям.
    rng = np.random.default_rng(SEED)
    shuffled = rng.permutation(active_users)
    cumulative = np.cumsum(user_counts.loc[shuffled].to_numpy())
    n_take = int(np.searchsorted(cumulative, TARGET_RATINGS)) + 1
    selected_users = set(shuffled[:n_take].tolist())

    # Шаг 3. Оставляем оценки только выбранных пользователей и освобождаем память.
    ratings = ratings_full[ratings_full["userId"].isin(selected_users)].reset_index(drop=True)
    del ratings_full

    # Сохраняем подвыборку в кэш для последующих запусков.
    ratings.to_csv(SAMPLE_PATH, index=False)
    print(f"Подвыборка сохранена в кэш: {SAMPLE_PATH}")

print(f"\nИтоговая подвыборка: {len(ratings):,} оценок, "
      f"{ratings['userId'].nunique():,} пользователей, "
      f"{ratings['movieId'].nunique():,} фильмов")
ratings.head()

## 4. Разведочный (описательный) анализ данных — EDA

Цель — понять структуру и качество данных: распределение оценок, активность
пользователей и фильмов, разреженность матрицы, временные и жанровые эффекты,
характер genome-признаков. Выводы EDA определяют решения по предобработке и
выбору моделей.

In [ ]:
# --- Базовые статистики подвыборки ---
# Оцениваем масштаб задачи и разреженность матрицы оценок: эти величины
# объясняют, почему "в лоб" заполнить матрицу user × movie невозможно.

n_ratings = len(ratings)
n_users = ratings["userId"].nunique()
n_movies = ratings["movieId"].nunique()

# Разреженность = доля реально проставленных оценок от всех возможных пар
sparsity = n_ratings / (n_users * n_movies)

print(f"Число оценок:        {n_ratings:,}")
print(f"Уникальных юзеров:   {n_users:,}")
print(f"Уникальных фильмов:  {n_movies:,}")
print(f"Заполнено пар user×movie: {sparsity:.4%} "
      f"(разреженность матрицы — {1 - sparsity:.4%})")
print("\nОписательная статистика по оценкам:")
ratings["rating"].describe()

**Вывод.** Подвыборка включает 2 млн оценок от 9 289 пользователей по 17 776 фильмам.
Заполнено лишь ≈1,2 % всех возможных пар «пользователь × фильм» (разреженность ≈98,8 %),
поэтому напрямую восстановить матрицу оценок невозможно — необходимы модели со скрытыми
представлениями (эмбеддингами). Средняя оценка составляет 3,52 при медиане 3,5, то есть
пользователи в среднем оценивают фильмы выше середины шкалы.

In [ ]:
# --- Распределение оценок ---
# Оценки дискретны (шаг 0.5 в диапазоне 0.5–5.0). Смотрим, как часто
# встречается каждое значение: это показывает смещение пользователей
# в сторону высоких оценок и пригодится при выборе baseline.

rating_counts = ratings["rating"].value_counts(normalize=True).sort_index()

plt.bar(rating_counts.index, rating_counts.values, width=0.4, edgecolor="white")
plt.xlabel("Оценка")
plt.ylabel("Доля оценок")
plt.title("Распределение оценок")
plt.xticks(np.arange(0.5, 5.5, 0.5))
plt.show()

print("Доли значений оценок:")
print((rating_counts * 100).round(2).astype(str) + " %")
print(f"\nСредняя оценка: {ratings['rating'].mean():.3f}")
print(f"Медианная оценка: {ratings['rating'].median():.1f}")

**Вывод.** Распределение оценок смещено в сторону высоких значений: самые частые оценки —
4,0 (28 %) и 3,0 (21 %), тогда как на низкие оценки (≤2,0) приходится менее 13 %. Заметно,
что целые оценки выставляются чаще полуцелых (особенность поведения пользователей). Такой
дисбаланс означает, что даже тривиальный прогноз «средним» (≈3,5) даёт умеренную ошибку —
это и задаёт нижнюю планку качества для baseline.

In [ ]:
# --- Активность пользователей и популярность фильмов ---
# Проверяем гипотезу о "длинном хвосте": немного очень активных
# пользователей и популярных фильмов и много редких. Это ключевой фактор
# для разреженности и проблемы холодного старта.

ratings_per_user = ratings.groupby("userId").size()
ratings_per_movie = ratings.groupby("movieId").size()

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(ratings_per_user, bins=50, edgecolor="white", color="#4c72b0")
axes[0].set_yscale("log")  # лог-шкала: иначе хвост не виден
axes[0].set_xlabel("Число оценок у пользователя")
axes[0].set_ylabel("Число пользователей (log)")
axes[0].set_title("Активность пользователей")

axes[1].hist(ratings_per_movie, bins=50, edgecolor="white", color="#dd8452")
axes[1].set_yscale("log")
axes[1].set_xlabel("Число оценок у фильма")
axes[1].set_ylabel("Число фильмов (log)")
axes[1].set_title("Популярность фильмов")

plt.tight_layout()
plt.show()

print("Оценок на пользователя:")
print(ratings_per_user.describe().round(1))
print("\nОценок на фильм:")
print(ratings_per_movie.describe().round(1))

**Вывод.** Подтверждается эффект «длинного хвоста». У пользователей медиана составляет
124 оценки (минимум 50 — следствие фильтра активности), тогда как у фильмов половина имеет
менее 9 оценок, а четверть — не более двух. Малое число оценок у большинства фильмов
порождает проблему холодного старта и обосновывает дальнейшую фильтрацию редко оцениваемых
фильмов на этапе предобработки.

In [ ]:
# --- Временной разрез ---
# Смотрим, как менялись активность (число оценок) и средняя оценка по
# годам. Это показывает, есть ли временной сдвиг в данных, который стоит
# учитывать при разбиении train/test.

ratings["year"] = ratings["timestamp"].dt.year
year_stats = ratings.groupby("year")["rating"].agg(count="size", mean="mean")

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(year_stats.index, year_stats["count"], edgecolor="white", color="#4c72b0")
axes[0].set_xlabel("Год")
axes[0].set_ylabel("Число оценок")
axes[0].set_title("Число оценок по годам")

axes[1].plot(year_stats.index, year_stats["mean"], marker="o", color="#dd8452")
axes[1].set_xlabel("Год")
axes[1].set_ylabel("Средняя оценка")
axes[1].set_title("Средняя оценка по годам")

plt.tight_layout()
plt.show()

print(f"Период данных: {ratings['year'].min()} – {ratings['year'].max()}")
year_stats.round(3)

**Вывод.** Данные охватывают период 1995–2015 гг. Средняя оценка по годам остаётся
стабильной (колеблется в узком диапазоне ≈3,42–3,63) без выраженного тренда, а объём оценок
распределён неравномерно (заметные пики в 2000 и 2005 гг.). Отсутствие временного дрейфа
целевой переменной позволяет использовать случайное разбиение на train/val/test, не прибегая
к строгому разбиению по времени.

In [ ]:
# --- Анализ жанров ---
# Жанр — первый кандидат в контентные признаки. Проверяем, насколько
# часто встречается каждый жанр и различается ли по нему средняя оценка
# (если различается — жанр несёт полезный сигнал для модели).

# Присоединяем жанры фильмов к оценкам и разворачиваем список жанров
# ("Adventure|Comedy" -> две строки) для подсчёта по каждому жанру.
ratings_genres = (
    ratings.merge(movies[["movieId", "genres"]], on="movieId", how="left")
    .assign(genre=lambda d: d["genres"].str.split("|"))
    .explode("genre")
)

genre_stats = (
    ratings_genres.groupby("genre")["rating"]
    .agg(count="size", mean="mean")
    .sort_values("count", ascending=False)
)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].barh(genre_stats.index, genre_stats["count"], edgecolor="white", color="#4c72b0")
axes[0].invert_yaxis()
axes[0].set_xlabel("Число оценок")
axes[0].set_title("Частота жанров")

# Средняя оценка по жанру: сортируем, чтобы увидеть "любимые" и "нелюбимые"
genre_by_mean = genre_stats.sort_values("mean")
axes[1].barh(genre_by_mean.index, genre_by_mean["mean"], edgecolor="white", color="#55a868")
axes[1].set_xlabel("Средняя оценка")
axes[1].set_xlim(genre_by_mean["mean"].min() - 0.1, genre_by_mean["mean"].max() + 0.1)
axes[1].set_title("Средняя оценка по жанрам")

plt.tight_layout()
plt.show()

genre_stats.round(3)

**Вывод.** Самые массовые жанры — Drama, Comedy и Action. При этом средняя оценка заметно
зависит от жанра: выше всего оцениваются Film-Noir (3,97), War (3,80) и Documentary (3,74),
ниже всего — Horror (3,27) и Children (3,40). Разброс средних оценок между жанрами достигает
≈0,7 балла, следовательно жанр несёт полезный сигнал и оправдан как контентный признак
гибридной модели.

In [ ]:
# --- Обзор genome-признаков ---
# genome_scores задаёт для каждого фильма вектор релевантности по ~1128
# тегам (значения 0..1). Это богатый источник контентных признаков, но
# их много — оценим их характер, чтобы понять, какие пригодны для отбора.

# Статистика релевантности по каждому тегу: средняя и разброс (std).
# Высокий std означает, что тег хорошо разделяет фильмы между собой.
tag_stats = (
    genome_scores.groupby("tagId")["relevance"]
    .agg(mean="mean", std="std")
    .merge(genome_tags, on="tagId")
)
print(f"Всего genome-тегов: {len(tag_stats)}")

# Распределение релевантности (по случайной подвыборке строк для скорости)
sample = genome_scores["relevance"].sample(200_000, random_state=SEED)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(sample, bins=50, edgecolor="white", color="#4c72b0")
axes[0].set_xlabel("Релевантность тега")
axes[0].set_ylabel("Число пар (фильм, тег)")
axes[0].set_title("Распределение релевантности genome-тегов")

# Топ-15 наиболее "различающих" тегов (по std) — кандидаты в признаки
top_var = tag_stats.sort_values("std", ascending=False).head(15)
axes[1].barh(top_var["tag"], top_var["std"], edgecolor="white", color="#8172b3")
axes[1].invert_yaxis()
axes[1].set_xlabel("Стандартное отклонение релевантности")
axes[1].set_title("Топ-15 тегов по разбросу релевантности")

plt.tight_layout()
plt.show()

top_var[["tag", "mean", "std"]].round(3)

**Вывод.** Распределение genome-релевантности сильно скошено к нулю: для большинства пар
«фильм–тег» тег нерелевантен. Наибольший разброс (std) имеют как жанровые теги (comedy,
action, horror, drama), так и тематические (tense, relationships, obsession) — именно они
лучше всего различают фильмы. Поскольку из 1128 тегов информативна лишь часть, перед
обучением гибридной модели потребуется отбор значимых признаков (раздел 6).

In [ ]:
# --- Год выпуска фильма (≠ год оценки) ---
# ВАЖНО: год выпуска извлекается из названия фильма и отражает "возраст"
# фильма. Это НЕ год выставления оценки (timestamp из раздела выше).
# Год выпуска — потенциальный контентный признак: классика может цениться
# выше новинок. Проверяем связь года выпуска со средней оценкой.

# Год указан в названии в скобках, напр. "Toy Story (1995)"
movies_year = movies.copy()
movies_year["release_year"] = (
    movies_year["title"].str.extract(r"\((\d{4})\)").astype("float")
)
print("Фильмов без распознанного года выпуска:",
      int(movies_year["release_year"].isna().sum()))

# Присоединяем год выпуска к оценкам подвыборки
ratings_release = ratings.merge(
    movies_year[["movieId", "release_year"]], on="movieId", how="left"
)

# Средняя оценка и число оценок в зависимости от года выпуска фильма
release_stats = (
    ratings_release.groupby("release_year")["rating"].agg(count="size", mean="mean")
)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].bar(release_stats.index, release_stats["count"], edgecolor="white", color="#4c72b0")
axes[0].set_xlabel("Год выпуска фильма")
axes[0].set_ylabel("Число оценок")
axes[0].set_title("Число оценок по году выпуска")

axes[1].plot(release_stats.index, release_stats["mean"], marker=".", color="#dd8452")
axes[1].set_xlabel("Год выпуска фильма")
axes[1].set_ylabel("Средняя оценка")
axes[1].set_title("Средняя оценка по году выпуска")

plt.tight_layout()
plt.show()

print(f"Диапазон годов выпуска: {release_stats.index.min():.0f} – {release_stats.index.max():.0f}")
release_stats.round(3)

Видна чёткая закономерность: фильмы 1930–1970-х оцениваются выше всего (≈3,8–4,0), после чего средняя оценка плавно снижается к новинкам 1990–2010-х (≈3,4–3,5) — эффект «классики». Большинство оценок приходится на фильмы 1990–2005 гг. (пик около 1995). Важно: у ранних фильмов (до 1920 г.) оценок мало, поэтому их средние шумны (разброс 2,0–4,0). Поскольку зависимость монотонна и выражена, год выпуска — сильный контентный признак, его стоит включить в гибридную модель (с нормализацией).

In [ ]:
# --- Покрытие genome-признаками ---
# Не у всех фильмов есть genome-вектор. Оцениваем, какая доля фильмов
# подвыборки (и какая доля оценок) обеспечена genome-признаками — это
# определяет, насколько "гибридной" может быть модель и нужна ли импутация
# для фильмов без genome.

movies_with_genome = set(genome_scores["movieId"].unique())

# Уникальные фильмы подвыборки и их покрытие genome
sub_movies = ratings["movieId"].unique()
covered_movies = np.isin(sub_movies, list(movies_with_genome))

# Доля оценок, относящихся к фильмам с genome-вектором
covered_ratings = ratings["movieId"].isin(movies_with_genome)

print(f"Всего фильмов с genome-вектором (в датасете): {len(movies_with_genome):,}")
print(f"\nФильмов в подвыборке: {len(sub_movies):,}")
print(f"  с genome-вектором:  {covered_movies.sum():,} ({covered_movies.mean():.1%})")
print(f"  без genome-вектора: {(~covered_movies).sum():,} ({(~covered_movies).mean():.1%})")
print(f"\nОценок в подвыборке: {len(ratings):,}")
print(f"  покрыто genome:     {covered_ratings.sum():,} ({covered_ratings.mean():.1%})")

ключевой практический результат: genome-вектор есть лишь у 57,9 % фильмов подвыборки, но они покрывают 99,0 % всех оценок. Это значит, что фильмы без genome — это редкие фильмы из «длинного хвоста», и гибридная модель сможет опираться на контентные признаки практически для всех взаимодействий. Для оставшегося ~1 % оценок (фильмы без genome) достаточно импутации нулевым вектором — отдельная сложная обработка не нужна.

In [ ]:
# --- Проверка качества данных ---
# Проверяем пропуски и дубликаты. Повторные оценки одной пары
# (пользователь, фильм) исказили бы обучение и могли бы вызвать утечку
# при разбиении на train/val/test, поэтому их важно выявить заранее.

print("Пропуски по столбцам подвыборки оценок:")
print(ratings.isna().sum())

# Дубликаты пар (userId, movieId) — один пользователь не должен оценивать
# один фильм дважды
dup_pairs = ratings.duplicated(subset=["userId", "movieId"]).sum()
print(f"\nДубликаты пар (userId, movieId): {dup_pairs:,}")
print(f"Полные дубликаты строк:          {ratings.duplicated().sum():,}")

# Контроль диапазона оценок (ожидаем 0.5–5.0)
print(f"\nДиапазон оценок: {ratings['rating'].min()} – {ratings['rating'].max()}")

данные чистые: пропусков нет, дубликатов пар (userId, movieId) нет, диапазон оценок корректен (0,5–5,0)

### Выводы по EDA

Проведённый разведочный анализ позволяет сформулировать ключевые наблюдения и решения для
последующих этапов работы:

1. **Качество данных.** Подвыборка не содержит пропусков и дубликатов пар
   «пользователь–фильм», оценки лежат в ожидаемом диапазоне [0,5; 5,0]. Дополнительная
   очистка не требуется, можно сразу переходить к формированию признаков.
2. **Разреженность.** Заполнено лишь ≈1,2 % пар «пользователь × фильм», поэтому задача
   решается моделями со скрытыми представлениями (эмбеддингами), а не прямым восстановлением
   матрицы оценок.
3. **Дисбаланс оценок.** Оценки смещены в сторону высоких значений (среднее ≈3,52, пик на 4,0).
   Прогноз «средним» задаёт нижнюю планку качества, относительно которой оцениваются модели.
4. **Длинный хвост.** Большинство фильмов имеют мало оценок (медиана 9), что создаёт проблему
   холодного старта и обосновывает фильтрацию редких фильмов на этапе предобработки.
5. **Стабильность во времени.** Средняя оценка по годам выставления не имеет выраженного
   тренда — допустимо случайное разбиение на обучающую, валидационную и тестовую выборки.
6. **Год выпуска как признак.** В отличие от года оценки, год выпуска фильма заметно связан
   со средней оценкой (старые фильмы ценятся выше, спад к новинкам), поэтому включается
   в число контентных признаков.
7. **Информативность контента.** Жанры (разброс средних оценок до ≈0,7 балла) и genome-теги
   несут полезный сигнал, что подтверждает целесообразность гибридного подхода. При этом из
   1128 genome-тегов информативна лишь часть — необходим отбор значимых признаков.
8. **Покрытие genome.** Genome-вектор имеют 57,9 % фильмов подвыборки, но они охватывают
   99,0 % оценок; фильмы без genome относятся к «длинному хвосту» и обрабатываются импутацией
   нулевым вектором.

Эти выводы определяют дальнейшие шаги: фильтрацию и переиндексацию данных, формирование
контентных признаков (жанры, год выпуска, genome), отбор значимых признаков и построение
моделей нарастающей сложности.

## 5. Предобработка данных

Готовим данные к обучению. Шаги раздела:

1. **Фильтрация редких фильмов** — отсекаем «длинный хвост» (мало оценок → шум и
   проблема холодного старта).
2. **Переиндексация id** — `userId`/`movieId` → непрерывные индексы `[0..N)`,
   необходимые для слоёв эмбеддингов.
3. **Контентные признаки фильма** — жанры (multi-hot), год выпуска (нормализованный)
   и матрица genome-релевантности. Признаки храним **на уровне фильма** (по `movie_idx`)
   и подставляем по индексу прямо в батче (раздел 7) — это избавляет от избыточного
   дублирования признаков на каждую из 2 млн оценок (вместо «тяжёлого» join всех таблиц).
4. **Разбиение train / val / test** — случайное (EDA не выявил временного дрейфа).

In [ ]:
# --- Фильтрация редко оцениваемых фильмов ---
# EDA показал "длинный хвост": у половины фильмов меньше 9 оценок. Такие
# фильмы дают ненадёжный сигнал и усиливают проблему холодного старта.
# Оставляем фильмы с числом оценок >= MIN_MOVIE_RATINGS. (Пользователи уже
# отфильтрованы по активности в разделе 3.)

movie_counts = ratings["movieId"].value_counts()
popular_movies = movie_counts.index[movie_counts >= MIN_MOVIE_RATINGS]

print(f"До фильтрации:  {len(ratings):,} оценок, "
      f"{ratings['movieId'].nunique():,} фильмов")

ratings = ratings[ratings["movieId"].isin(popular_movies)].reset_index(drop=True)

print(f"После фильтрации (>= {MIN_MOVIE_RATINGS} оценок у фильма): "
      f"{len(ratings):,} оценок, {ratings['movieId'].nunique():,} фильмов, "
      f"{ratings['userId'].nunique():,} пользователей")

In [ ]:
# --- Переиндексация идентификаторов ---
# Слои эмбеддингов в PyTorch требуют непрерывные индексы [0..N). Исходные
# userId/movieId разрежены (с пропусками), поэтому строим словари
# соответствия "исходный id -> индекс" (и обратные — для интерпретации).

unique_users = ratings["userId"].unique()
unique_movies = ratings["movieId"].unique()

user2idx = {uid: i for i, uid in enumerate(unique_users)}
movie2idx = {mid: i for i, mid in enumerate(unique_movies)}
idx2user = {i: uid for uid, i in user2idx.items()}
idx2movie = {i: mid for mid, i in movie2idx.items()}

# Добавляем колонки с непрерывными индексами
ratings["user_idx"] = ratings["userId"].map(user2idx).astype("int32")
ratings["movie_idx"] = ratings["movieId"].map(movie2idx).astype("int32")

n_users = len(user2idx)
n_movies = len(movie2idx)
print(f"Пользователей (n_users): {n_users:,}")
print(f"Фильмов (n_movies):      {n_movies:,}")
ratings[["userId", "user_idx", "movieId", "movie_idx", "rating"]].head()

In [ ]:
# --- Контентные признаки фильма: жанры и год выпуска ---
# Признаки формируем НА УРОВНЕ ФИЛЬМА (один вектор на movie_idx), а не на
# уровне каждой оценки: иначе признаки фильма дублировались бы для всех его
# оценок. В модель они подставляются по movie_idx уже в батче (раздел 7).

# Таблица оставшихся фильмов, упорядоченная по movie_idx (строка i <-> индекс i)
movies_kept = movies[movies["movieId"].isin(movie2idx)].copy()
movies_kept["movie_idx"] = movies_kept["movieId"].map(movie2idx)
movies_kept = movies_kept.sort_values("movie_idx").reset_index(drop=True)

# (1) Жанры -> multi-hot. У фильма может быть несколько жанров
# ("Action|Comedy"), поэтому используем get_dummies с разделителем "|".
genres_df = movies_kept["genres"].str.get_dummies("|")
# Метка-заглушка "(no genres listed)" не несёт информации — убираем при наличии.
genres_df = genres_df.drop(columns=["(no genres listed)"], errors="ignore")
genre_cols = genres_df.columns.tolist()
genres_mat = genres_df.to_numpy(dtype="float32")

# (2) Год выпуска из названия (regex, как в EDA). Пропуски заполняем медианой,
# затем нормируем (z-score): нейросети устойчивее обучаются на
# стандартизованных входах.
release_year = movies_kept["title"].str.extract(r"\((\d{4})\)")[0].astype("float")
release_year = release_year.fillna(release_year.median())
year_vec = ((release_year - release_year.mean()) / release_year.std()).to_numpy(dtype="float32")

print(f"Жанровых признаков: {len(genre_cols)}")
print(f"genres_mat: {genres_mat.shape}, year_vec: {year_vec.shape}")
print("Жанры:", ", ".join(genre_cols))

In [ ]:
# --- Контентные признаки фильма: матрица genome-релевантности ---
# genome задаёт для фильма вектор из 1128 тегов (релевантность 0..1) — самый
# богатый контентный источник. Строим плотную матрицу movie_idx x 1128.
# Фильмы без genome (редкие, см. EDA) получают нулевой вектор (импутация) —
# отдельная обработка не требуется.

# Берём genome только для оставшихся фильмов и переводим в индексы movie_idx.
genome_kept = genome_scores[genome_scores["movieId"].isin(movie2idx)].copy()
genome_kept["movie_idx"] = genome_kept["movieId"].map(movie2idx)

# pivot: строки — movie_idx, столбцы — tagId, значения — relevance.
genome_pivot = genome_kept.pivot(index="movie_idx", columns="tagId", values="relevance")

# Выравниваем по всем фильмам [0..n_movies) и всем тегам (фиксированный порядок).
# Отсутствующие строки/значения (фильмы без genome) заполняются нулём.
tag_cols = np.sort(genome_scores["tagId"].unique())
genome_pivot = genome_pivot.reindex(index=range(n_movies), columns=tag_cols)
genome_mat = np.nan_to_num(genome_pivot.to_numpy(dtype="float32"), nan=0.0)

covered = int((genome_mat.sum(axis=1) > 0).sum())
print(f"genome_mat: {genome_mat.shape}")
print(f"Фильмов с genome-вектором: {covered:,} из {n_movies:,} "
      f"({covered / n_movies:.1%}); остальные — нулевой вектор (импутация)")

In [ ]:
# --- Разбиение на train / val / test ---
# EDA не выявил временного дрейфа средней оценки, поэтому используем случайное
# разбиение по взаимодействиям (а не строго по времени). Сначала отделяем
# test, затем из остатка выделяем валидацию. Индексы user_idx/movie_idx
# построены на всём отфильтрованном наборе, поэтому почти все пользователи и
# фильмы присутствуют в train (минимум проблем холодного старта).

train_val_df, test_df = train_test_split(
    ratings, test_size=TEST_SIZE, random_state=SEED, shuffle=True
)
# Долю валидации пересчитываем относительно остатка после изъятия test.
val_relative = VAL_SIZE / (1.0 - TEST_SIZE)
train_df, val_df = train_test_split(
    train_val_df, test_size=val_relative, random_state=SEED, shuffle=True
)

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"train: {len(train_df):,} ({len(train_df) / len(ratings):.0%})")
print(f"val:   {len(val_df):,} ({len(val_df) / len(ratings):.0%})")
print(f"test:  {len(test_df):,} ({len(test_df) / len(ratings):.0%})")
print(f"Сумма: {len(train_df) + len(val_df) + len(test_df):,} "
      f"(должна совпадать с {len(ratings):,})")

# Контроль cold-start: сколько пользователей/фильмов из test не встречаются в train.
train_users, train_movies = set(train_df["user_idx"]), set(train_df["movie_idx"])
cold_users = len(set(test_df["user_idx"]) - train_users)
cold_movies = len(set(test_df["movie_idx"]) - train_movies)
print(f"\nНовых (cold-start) в test: пользователей — {cold_users}, фильмов — {cold_movies}")

### Выводы по предобработке данных

**Фильтрация.** Отсев фильмов с менее 20 оценками убрал ~10 870 «хвостовых» фильмов
(61 % от общего числа), сократив объём выборки лишь на ~2,6 % (с 2 000 277 до
1 947 709 оценок). Это подтверждает вывод EDA: редкие фильмы дают мало оценок, но
занимают большую часть пространства фильмов. После фильтрации: 9 289 пользователей,
6 906 фильмов.

**Переиндексация.** Идентификаторы приведены к непрерывным индексам [0..6905] и
[0..9288] — необходимое условие для слоёв эмбеддингов PyTorch.

**Контентные признаки** сформированы на уровне фильма (6 906 строк): multi-hot жанры
(20 признаков), нормализованный год выпуска (1 признак), genome-матрица
(6 906 × 1 128 тегов) с нулевой импутацией для фильмов без genome.

**Разбиение 80/10/10.**

## 6. Отбор значимых признаков

Среди контентных признаков отбор реально нужен прежде всего для **genome-тегов**
(их 1128): большинство тегов нерелевантны для большинства фильмов, а такая
размерность раздула бы вход гибридной модели и добавила бы шум. Жанры (19) и год (1)
оставляем целиком — их мало и они интерпретируемы.

Шаги отбора (применяются к genome-тегам):

1. **VarianceThreshold** — убираем почти постоянные теги (не различают фильмы).
2. **Значимость относительно оценки** — корреляция релевантности тега со средней
   оценкой фильма (по train, без утечки) → отбор top-K тегов.
3. **Сборка итоговой матрицы** контентных признаков (жанры + год + отобранные теги),
   которая пойдёт в гибридную модель (раздел 8).

In [ ]:
# --- Шаг 1: удаление низковариативных genome-тегов ---
# Отбор применяем ТОЛЬКО к genome-тегам. Многие теги почти постоянны
# (нерелевантны большинству фильмов) — они не помогают различать фильмы,
# поэтому отсекаем их по дисперсии.

from sklearn.feature_selection import VarianceThreshold

var_selector = VarianceThreshold(threshold=0.005)
genome_var = var_selector.fit_transform(genome_mat)
var_mask = var_selector.get_support()   # маска оставленных тегов
tagid_var = tag_cols[var_mask]          # их исходные tagId

print(f"genome-тегов было:                {genome_mat.shape[1]}")
print(f"осталось после VarianceThreshold: {genome_var.shape[1]} (порог 0.005)")

In [ ]:
# --- Шаги 2–3: значимость тегов, отбор top-K и сборка матрицы признаков ---
# Оцениваем связь каждого тега (после VarianceThreshold) с целевой переменной:
# считаем среднюю оценку фильма ПО TRAIN (без утечки из val/test) и берём
# |корреляцию| релевантности тега с ней. Оставляем top-K тегов, визуализируем
# их и собираем итоговую матрицу контентных признаков для гибридной модели.

TOP_K_TAGS = 100  # сколько genome-тегов оставить

# Средняя оценка фильма по обучающей выборке (индекс — movie_idx)
movie_mean = train_df.groupby("movie_idx")["rating"].mean().reindex(range(n_movies))

# Теги после VarianceThreshold в виде DataFrame (столбцы — tagId)
genome_var_df = pd.DataFrame(genome_var, columns=tagid_var)

# |Корреляция Пирсона| каждого тега со средней оценкой фильма -> top-K
corr = genome_var_df.corrwith(movie_mean).abs().dropna()
top_tags = corr.sort_values(ascending=False).head(TOP_K_TAGS)
selected_tagid = top_tags.index.to_numpy()

print(f"genome-тегов после VarianceThreshold: {genome_var.shape[1]}")
print(f"отобрано top-K: {len(selected_tagid)} "
      f"(|корреляция| {top_tags.min():.3f} – {top_tags.max():.3f})")

# Топ-20 тегов с названиями — для наглядности
top20 = top_tags.head(20).rename_axis("tagId").reset_index(name="corr")
top20["tagId"] = top20["tagId"].astype(int)
top20 = top20.merge(genome_tags, on="tagId")

plt.figure(figsize=(8, 6))
plt.barh(top20["tag"], top20["corr"], edgecolor="white", color="#2a9d8f")
plt.gca().invert_yaxis()
plt.xlabel("|Корреляция| со средней оценкой фильма")
plt.title("Топ-20 значимых genome-тегов")
plt.tight_layout()
plt.show()

# Итоговая матрица признаков: жанры (19) + год (1) + отобранные теги (K).
sel_mask = np.isin(tag_cols, selected_tagid)
genome_selected = genome_mat[:, sel_mask]

content_mat = np.hstack([
    genres_mat,                  # жанры (multi-hot)
    year_vec.reshape(-1, 1),     # нормализованный год выпуска
    genome_selected,             # отобранные genome-теги
]).astype("float32")

content_cols = genre_cols + ["release_year"] + [f"tag_{t}" for t in tag_cols[sel_mask]]
print(f"Итоговая матрица контентных признаков: {content_mat.shape}")
print(f"  жанры: {len(genre_cols)}, год: 1, genome-теги: {int(sel_mask.sum())}")

### Итоговый набор признаков

Финальный набор контентных признаков фильма (матрица `content_mat`, форма 6906 × 120):

| Группа | Кол-во | Описание | Отбор |
|--------|--------|----------|-------|
| Жанры | 19 | multi-hot кодирование жанров фильма | взяты целиком |
| Год выпуска | 1 | год из названия, нормализован (z-score) | взят целиком |
| Genome-теги | 100 | top-K тегов по \|корреляции\| со средней оценкой | 1128 → 899 (VarianceThreshold) → 100 |
| **Итого** | **120** | | |

Признаки хранятся на уровне фильма (по `movie_idx`) и подставляются в модель по
индексу фильма в батче. Этот набор используется **только гибридной моделью**
(раздел 8); модели MF и NeuMF работают без контентных признаков — это позволит в
сравнительном анализе оценить вклад контентной информации.

### Выводы по отбору признаков

Отбор признаков применялся целенаправленно к genome-тегам (1128 шт.); жанры (19) и
год выпуска (1) оставлены целиком ввиду малочисленности и интерпретируемости.

- **VarianceThreshold** (порог 0.005) убрал 229 почти постоянных тегов: осталось 899.
  Это подтверждает вывод EDA — значительная часть тегов нерелевантна большинству
  фильмов и не помогает их различать.
- **Отбор по значимости.** Из 899 тегов выбраны top-100 с наибольшей |корреляцией|
  релевантности со средней оценкой фильма (по обучающей выборке, без утечки). Диапазон
  |корреляции| отобранных тегов — 0.473–0.783, то есть все они сохраняют выраженную
  связь с целевой переменной.
- **Характер отобранных тегов.** Наиболее значимы оценочные теги, отражающие качество
  фильма («predictable», «bad plot», «overrated», «movielens top pick», «boring»),
  что закономерно: именно восприятие качества сильнее всего связано с оценкой.
- **Итоговый набор** контентных признаков фильма — 120 признаков (19 жанров +
  1 год + 100 genome-тегов), оформленных в матрицу `content_mat` (6906 × 120).
  Эта матрица подаётся в гибридную модель (раздел 8).

Поскольку даже у 100-го тега |корреляция| остаётся высокой (0.473), число тегов
выбрано с запасом компактности; при необходимости его можно увеличить или заменить
фиксированное K порогом по корреляции.

## 7. Проектирование и реализация исследовательского стенда

Единый стенд обеспечивает честное сравнение моделей: общий формат данных
(`Dataset`/`DataLoader`), единый цикл обучения с ранней остановкой, единые функции
метрик и общий интерфейс оценки. Все модели обучаются и оцениваются одинаково.

In [ ]:
# --- Формат данных: Dataset и DataLoader ---
# Все модели обучаются на одинаковых батчах (user_idx, movie_idx) -> rating.
# Контентные признаки фильма в батч НЕ кладём: иначе 120-мерный вектор
# дублировался бы для каждой из ~1.5 млн оценок. Вместо этого храним матрицу
# признаков один раз как тензор и обращаемся к ней по movie_idx уже внутри
# гибридной модели (раздел 8) — это экономит память.

BATCH_SIZE = 1024  # компромисс между скоростью обучения и стабильностью градиента

# Матрица контентных признаков (раздел 6) — единый тензор на устройстве.
content_tensor = torch.tensor(content_mat, dtype=torch.float32, device=device)
content_dim = content_tensor.shape[1]


class RatingDataset(Dataset):
    """Хранит тройки (пользователь, фильм, оценка) в виде тензоров."""

    def __init__(self, df):
        self.users = torch.tensor(df["user_idx"].to_numpy(), dtype=torch.long)
        self.movies = torch.tensor(df["movie_idx"].to_numpy(), dtype=torch.long)
        self.ratings = torch.tensor(df["rating"].to_numpy(), dtype=torch.float32)

    def __len__(self):
        return len(self.ratings)

    def __getitem__(self, i):
        return self.users[i], self.movies[i], self.ratings[i]


def make_loader(df, batch_size=BATCH_SIZE, shuffle=False):
    """Удобная обёртка для создания DataLoader по выборке."""
    return DataLoader(RatingDataset(df), batch_size=batch_size, shuffle=shuffle)


# Загрузчики для трёх выборок (train перемешиваем, val/test — нет).
train_loader = make_loader(train_df, shuffle=True)
val_loader = make_loader(val_df)
test_loader = make_loader(test_df)

print(f"Размер батча: {BATCH_SIZE}")
print(f"Батчей: train {len(train_loader)}, val {len(val_loader)}, test {len(test_loader)}")
print(f"Тензор контентных признаков: {tuple(content_tensor.shape)} на устройстве {device}")

In [ ]:
# --- Единые функции метрик ---
# Используем одни и те же метрики для всех моделей и baseline, чтобы
# сравнение было честным. RMSE — основная (сильнее штрафует крупные ошибки),
# MAE — средняя абсолютная ошибка, R² — доля объяснённой дисперсии.

def compute_metrics(y_true, y_pred):
    """Возвращает словарь {RMSE, MAE, R2} по истинным и предсказанным оценкам."""
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    mae = mean_absolute_error(y_true, y_pred)
    r2 = r2_score(y_true, y_pred)
    return {"RMSE": rmse, "MAE": mae, "R2": r2}

In [ ]:
# --- Универсальные процедуры обучения и оценки ---
# Все нейросетевые модели (раздел 8) имеют единый интерфейс forward(users, movies)
# и обучаются одной и той же процедурой — это гарантирует сопоставимость
# результатов. Оценки обрезаем до диапазона [0.5, 5.0] (за его пределами оценок
# не бывает, обрезка немного снижает ошибку).

RATING_MIN, RATING_MAX = 0.5, 5.0


@torch.no_grad()
def evaluate_model(model, loader):
    """Прогон модели по выборке без обучения; возвращает словарь метрик."""
    model.eval()
    preds, trues = [], []
    for users, movies, ratings_b in loader:
        out = model(users.to(device), movies.to(device)).cpu().numpy()
        preds.append(out)
        trues.append(ratings_b.numpy())
    y_pred = np.clip(np.concatenate(preds), RATING_MIN, RATING_MAX)
    y_true = np.concatenate(trues)
    return compute_metrics(y_true, y_pred)


def train_model(model, train_loader, val_loader, *, epochs=20, lr=1e-3,
                weight_decay=0.0, patience=3, verbose=True):
    """Обучает модель с ранней остановкой по RMSE на валидации.

    Возвращает историю обучения (RMSE по эпохам). По окончании в модель
    загружаются веса лучшей эпохи (минимальный val RMSE).
    """
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)
    criterion = nn.MSELoss()

    history = {"train_rmse": [], "val_rmse": []}
    best_val, best_state, bad_epochs = float("inf"), None, 0

    for epoch in range(1, epochs + 1):
        model.train()
        sq_err, n = 0.0, 0
        for users, movies, ratings_b in train_loader:
            users, movies = users.to(device), movies.to(device)
            ratings_b = ratings_b.to(device)

            optimizer.zero_grad()
            pred = model(users, movies)
            loss = criterion(pred, ratings_b)
            loss.backward()
            optimizer.step()

            sq_err += loss.item() * len(ratings_b)
            n += len(ratings_b)

        train_rmse = (sq_err / n) ** 0.5
        val_rmse = evaluate_model(model, val_loader)["RMSE"]
        history["train_rmse"].append(train_rmse)
        history["val_rmse"].append(val_rmse)
        if verbose:
            print(f"эпоха {epoch:2d} | train RMSE {train_rmse:.4f} | val RMSE {val_rmse:.4f}")

        # Ранняя остановка: если val RMSE не улучшается patience эпох подряд.
        if val_rmse < best_val - 1e-4:
            best_val = val_rmse
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= patience:
                if verbose:
                    print(f"Ранняя остановка на эпохе {epoch} (нет улучшения {patience} эпох)")
                break

    if best_state is not None:
        model.load_state_dict(best_state)  # возвращаем лучшие веса
    return history

In [ ]:
# --- Baseline-предсказатели (точка отсчёта) ---
# Простые модели без эмбеддингов, относительно которых оцениваем нейросети.
# Все статистики считаем ТОЛЬКО по train (без утечки), метрики — на test.

# Единый словарь результатов: сюда будем складывать метрики всех моделей
# для итогового сравнения (раздел 11).
results = {}

y_test = test_df["rating"].to_numpy()

# Baseline 1. Глобальное среднее: предсказываем всем одну константу mu.
mu = train_df["rating"].mean()
pred_mean = np.full(len(test_df), mu)
results["Global mean"] = compute_metrics(y_test, np.clip(pred_mean, RATING_MIN, RATING_MAX))

# Baseline 2. Модель смещений: r = mu + b_movie + b_user.
# b_movie — отклонение средней оценки фильма от mu (популярность/качество);
# b_user — отклонение пользователя от mu с учётом уже учтённого b_movie
# (строгость/доброта оценщика).
movie_bias = train_df.groupby("movie_idx")["rating"].mean() - mu
resid = train_df["rating"] - mu - train_df["movie_idx"].map(movie_bias).to_numpy()
user_bias = resid.groupby(train_df["user_idx"]).mean()

pred_bias = (
    mu
    + test_df["movie_idx"].map(movie_bias).fillna(0).to_numpy()
    + test_df["user_idx"].map(user_bias).fillna(0).to_numpy()
)
results["Bias (mu+b_u+b_i)"] = compute_metrics(y_test, np.clip(pred_bias, RATING_MIN, RATING_MAX))

print(f"Глобальное среднее train: mu = {mu:.3f}\n")
print("Метрики baseline на test:")
print(pd.DataFrame(results).T.round(4))

## 8. Нейросетевые модели

Три архитектуры нарастающей сложности. Различие — в том, как используется информация:
от чистой коллаборативной (MF, NeuMF) к гибридной (контент + взаимодействия).

In [ ]:
# --- Модель 1: Matrix Factorization (MF) ---
# Базовая коллаборативная модель. Оценка = скалярное произведение латентных
# векторов пользователя и фильма + их смещения + глобальное среднее:
#   r = mu + b_u + b_i + <p_u, q_i>
# Смещения играют ту же роль, что в baseline (раздел 7), а скалярное
# произведение добавляет персонализацию (линейное взаимодействие).

class MatrixFactorization(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=32, global_mean=0.0):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        self.user_bias = nn.Embedding(n_users, 1)
        self.movie_bias = nn.Embedding(n_movies, 1)
        # стартуем около среднего по данным -> быстрее сходимость
        self.global_bias = nn.Parameter(torch.tensor(float(global_mean)))

        # малая инициализация эмбеддингов; смещения — с нуля
        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.movie_emb.weight, std=0.05)
        nn.init.zeros_(self.user_bias.weight)
        nn.init.zeros_(self.movie_bias.weight)

    def forward(self, users, movies):
        dot = (self.user_emb(users) * self.movie_emb(movies)).sum(dim=1)
        return (dot
                + self.user_bias(users).squeeze(1)
                + self.movie_bias(movies).squeeze(1)
                + self.global_bias)

In [ ]:
# --- Модель 2: NeuMF (Neural Collaborative Filtering) ---
# Развитие MF: линейное скалярное произведение заменяется обучаемым
# НЕЛИНЕЙНЫМ взаимодействием. По статье NCF модель объединяет две ветви:
#   - GMF: поэлементное произведение эмбеддингов (обобщённая MF);
#   - MLP: конкатенация эмбеддингов -> многослойный персептрон.
# Их выходы объединяются и сводятся в одну оценку. Так проверяем вклад
# нелинейности по сравнению с MF.

def make_mlp(in_dim, hidden, dropout):
    """Собирает MLP (Linear + ReLU + Dropout) по списку ширины слоёв hidden.
    Используется в NeuMF и Hybrid."""
    layers, prev = [], in_dim
    for h in hidden:
        layers += [nn.Linear(prev, h), nn.ReLU(), nn.Dropout(dropout)]
        prev = h
    return nn.Sequential(*layers)


class NeuMF(nn.Module):
    def __init__(self, n_users, n_movies, emb_dim=32, hidden=(64, 32), dropout=0.2):
        super().__init__()
        # GMF-ветвь (линейное взаимодействие)
        self.gmf_user = nn.Embedding(n_users, emb_dim)
        self.gmf_movie = nn.Embedding(n_movies, emb_dim)
        # MLP-ветвь (нелинейное взаимодействие) — отдельные эмбеддинги
        self.mlp_user = nn.Embedding(n_users, emb_dim)
        self.mlp_movie = nn.Embedding(n_movies, emb_dim)
        self.mlp = make_mlp(2 * emb_dim, hidden, dropout)
        # объединение ветвей -> скалярная оценка
        self.head = nn.Linear(emb_dim + hidden[-1], 1)

        for emb in (self.gmf_user, self.gmf_movie, self.mlp_user, self.mlp_movie):
            nn.init.normal_(emb.weight, std=0.05)

    def forward(self, users, movies):
        gmf = self.gmf_user(users) * self.gmf_movie(movies)
        mlp_in = torch.cat([self.mlp_user(users), self.mlp_movie(movies)], dim=1)
        x = torch.cat([gmf, self.mlp(mlp_in)], dim=1)
        return self.head(x).squeeze(1)

In [ ]:
# --- Модель 3: Hybrid (коллаборатив + контент) ---
# К эмбеддингам пользователя и фильма добавляем КОНТЕНТНЫЕ признаки фильма
# (жанры + год + отобранные genome-теги, раздел 6). Признаки берутся из
# общего тензора content по индексу фильма (буфер, не обучается). Так модель
# может опираться и на взаимодействия, и на содержание фильма — это её
# главное отличие от MF/NeuMF.

class HybridModel(nn.Module):
    def __init__(self, n_users, n_movies, content, emb_dim=32,
                 hidden=(128, 64), dropout=0.2):
        super().__init__()
        self.user_emb = nn.Embedding(n_users, emb_dim)
        self.movie_emb = nn.Embedding(n_movies, emb_dim)
        # контентные признаки фильма (n_movies x content_dim) — фиксированы
        self.register_buffer("content", content)

        in_dim = 2 * emb_dim + content.shape[1]
        self.mlp = make_mlp(in_dim, hidden, dropout)
        self.head = nn.Linear(hidden[-1], 1)

        nn.init.normal_(self.user_emb.weight, std=0.05)
        nn.init.normal_(self.movie_emb.weight, std=0.05)

    def forward(self, users, movies):
        u = self.user_emb(users)
        m = self.movie_emb(movies)
        c = self.content[movies]          # контентные признаки по индексу фильма
        x = torch.cat([u, m, c], dim=1)
        return self.head(self.mlp(x)).squeeze(1)

## 9. Подбор гиперпараметров

Небольшой поиск (grid/random search) по валидационной выборке. Пространство поиска:
размер эмбеддинга, learning rate, число и ширина скрытых слоёв, dropout, weight decay.
Для каждой модели выбираем лучшую конфигурацию по RMSE на валидации.

In [ ]:
# Цикл подбора гиперпараметров для каждой модели:
# перебор конфигураций, обучение на train, оценка на val,
# сводная таблица результатов и выбор лучших гиперпараметров.


## 10. Обучение финальных моделей

In [ ]:
# Обучение baseline и трёх моделей с лучшими гиперпараметрами на train;
# сохранение истории обучения (loss/RMSE по эпохам).


In [ ]:
# Графики сходимости (train/val) для каждой модели.


## 11. Оценка качества и сравнительный анализ

Финальное сравнение моделей проводится на отложенной test-выборке, не участвовавшей
ни в обучении, ни в подборе гиперпараметров.

In [ ]:
# Сводная таблица метрик (RMSE / MAE / R²) по всем моделям и baseline.


In [ ]:
# Столбчатые диаграммы сравнения метрик между моделями.


In [ ]:
# Анализ остатков лучшей модели: распределение ошибок,
# ошибки по сегментам (активные/неактивные пользователи, популярные/редкие фильмы).


## 12. Выбор лучшей модели и анализ результатов

_(Обоснование выбора лучшей модели по метрикам; интерпретация: что дали контентные
признаки и их отбор, на каких объектах модели ошибаются, проблема cold-start.)_

## 13. Выводы

_(Итоги по решению задачи; сравнение методов; ограничения подхода и направления
дальнейшего развития.)_